In [ ]:
# Persistent paths on Unity Catalog Volumes (survives serverless restarts)
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"
EMBEDDING_DELTA_PATH = "/Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta"
EMBEDDING_TABLE_NAME = "workspace.default.legal_embeddings_test"

# NOTE: On Databricks serverless, SQLite over UC Volumes can throw disk I/O errors.
# Best-practice here: keep Delta as source-of-truth and use session-local Chroma index for fast retrieval tests.
CHROMA_INIT_MODE = "local_ephemeral"   # options: local_ephemeral, volume_attempt
CHROMA_DB_CANDIDATES = [
    "/Volumes/workspace/legal_data/chroma_db/",
    "/Volumes/workspace/legal_data/chroma_db/legal_knowledge_test",
    "/Volumes/workspace/legal_data/vector_db_test/chroma_db_legal_knowledge_test",
]
LOCAL_CHROMA_FALLBACK_PATH = "/local_disk0/tmp/chroma_db_legal_knowledge_test"

COLLECTION_NAME = "legal_knowledge"
PRIMARY_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FALLBACK_EMBED_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"

BATCH_SIZE = 128
FORCE_REBUILD_CHROMA = False
CHROMA_RESET_ON_TENANT_ERROR = True



In [ ]:
%pip install -q --upgrade sentence-transformers chromadb transformers accelerate


In [ ]:
import os
import json
import shutil
import traceback
from datetime import datetime, timezone

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F


CELL_DIAGNOSTICS = []


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def report_cell(cell_name: str, ok: bool, details: str):
    status = "OK" if ok else "FAIL"
    entry = {
        "cell": cell_name,
        "status": status,
        "details": details,
        "ts": datetime.now(timezone.utc).isoformat(),
    }
    CELL_DIAGNOSTICS.append(entry)
    print(f"[DIAG] {cell_name} -> {status}: {details}")


def print_diagnostics_summary():
    print("\n=== CELL DIAGNOSTICS SUMMARY ===")
    for d in CELL_DIAGNOSTICS:
        print(f"{d['cell']}: {d['status']} | {d['details']}")



In [ ]:
print("[CELL 3] START - Loading Gold dataset")

try:
    gold_df = spark.read.format("delta").load(GOLD_PATH)

    required_cols = ["chunk_id", "chunk_text", "act_name", "section_number", "category", "file_name"]
    missing_cols = [c for c in required_cols if c not in gold_df.columns]
    if missing_cols:
        raise ValueError(f"Gold table is missing required columns: {missing_cols}")

    gold_df = (
        gold_df.select(*required_cols)
        .dropna(subset=["chunk_id", "chunk_text"])
        .dropDuplicates(["chunk_id"])
    )

    gold_count = gold_df.count()
    if gold_count == 0:
        raise ValueError("Gold dataset is empty after filtering. Cannot build embeddings.")

    log(f"Gold chunks ready: {gold_count}")
    gold_df.show(10, truncate=120)
    report_cell("Cell 3 - Load Gold", True, f"rows={gold_count}")
    print(f"[CELL 3] END - Success, usable rows={gold_count}")

except Exception as e:
    report_cell("Cell 3 - Load Gold", False, str(e))
    print(f"[CELL 3] END - Failed: {e}")
    raise



In [ ]:
embedding_model = None
loaded_model_name = None

for model_name in [PRIMARY_EMBED_MODEL, FALLBACK_EMBED_MODEL]:
    try:
        log(f"Loading embedding model: {model_name}")
        embedding_model = SentenceTransformer(model_name)
        _ = embedding_model.encode(["health check"], show_progress_bar=False)
        loaded_model_name = model_name
        log(f"Embedding model loaded: {model_name}")
        break
    except Exception as e:
        log(f"Failed to load {model_name}: {e}")

if embedding_model is None:
    report_cell("Cell 4 - Load Embedding Model", False, "No embedding model could be loaded")
    raise RuntimeError("Could not load any embedding model. Check internet/HuggingFace access.")

report_cell("Cell 4 - Load Embedding Model", True, f"model={loaded_model_name}")



In [ ]:
print("[CELL 5] START - Initializing Chroma")

os.makedirs("/Volumes/workspace/legal_data/vector_db_test", exist_ok=True)
os.makedirs("/Volumes/workspace/legal_data/chroma_db", exist_ok=True)

CHROMA_DB_PATH = None
CHROMA_MODE = "disabled"
client = None
collection = None
init_errors = []


def expand_fs_paths(path):
    paths = [path]
    if path.startswith("/Volumes/"):
        paths.append("/dbfs" + path)

    out = []
    seen = set()
    for p in paths:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def try_init_chroma(path):
    local_client = chromadb.PersistentClient(
        path=path,
        settings=Settings(anonymized_telemetry=False, allow_reset=True),
    )

    if FORCE_REBUILD_CHROMA:
        try:
            local_client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass

    local_collection = local_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    # Validate embedding dimension compatibility
    sample = embedding_model.encode(["dimension probe"], show_progress_bar=False)[0]
    expected_dim = len(sample)

    if local_collection.count() > 0:
        probe = local_collection.get(limit=1, include=["embeddings"])
        existing_embs = probe.get("embeddings") or []
        if existing_embs and len(existing_embs[0]) != expected_dim:
            local_client.delete_collection(COLLECTION_NAME)
            local_collection = local_client.get_or_create_collection(
                name=COLLECTION_NAME,
                metadata={"hnsw:space": "cosine"}
            )

    return local_client, local_collection


def try_volume_candidates():
    global CHROMA_DB_PATH, client, collection
    for candidate in CHROMA_DB_CANDIDATES:
        for fs_path in expand_fs_paths(candidate):
            try:
                os.makedirs(fs_path, exist_ok=True)
                client, collection = try_init_chroma(fs_path)
                CHROMA_DB_PATH = fs_path
                return True
            except Exception as e:
                init_errors.append(f"{fs_path}: {e}")
    return False


if CHROMA_INIT_MODE == "local_ephemeral":
    try:
        os.makedirs(LOCAL_CHROMA_FALLBACK_PATH, exist_ok=True)
        client, collection = try_init_chroma(LOCAL_CHROMA_FALLBACK_PATH)
        CHROMA_DB_PATH = LOCAL_CHROMA_FALLBACK_PATH
        CHROMA_MODE = "local_ephemeral"
        report_cell("Cell 5 - Init Chroma", True, f"mode={CHROMA_MODE}, path={CHROMA_DB_PATH}, existing_vectors={collection.count()}")
        print(f"[CELL 5] END - Success (mode={CHROMA_MODE}, path={CHROMA_DB_PATH})")
    except Exception as e:
        init_errors.append(str(e))

elif CHROMA_INIT_MODE == "volume_attempt":
    ok = try_volume_candidates()
    if ok:
        CHROMA_MODE = "persistent_volume"
        report_cell("Cell 5 - Init Chroma", True, f"mode={CHROMA_MODE}, path={CHROMA_DB_PATH}, existing_vectors={collection.count()}")
        print(f"[CELL 5] END - Success (mode={CHROMA_MODE}, path={CHROMA_DB_PATH})")

if collection is None:
    CHROMA_MODE = "unavailable"
    report_cell(
        "Cell 5 - Init Chroma",
        False,
        "Chroma init failed. Delta embeddings will still be persisted. " + " | ".join(init_errors[:4])
    )
    print("[CELL 5] END - Chroma unavailable, continuing with Delta-only mode")

manifest_path = "/Volumes/workspace/legal_data/vector_db_test/embedding_runtime_manifest.json"
manifest = {
    "embedding_delta_path": EMBEDDING_DELTA_PATH,
    "embedding_table": EMBEDDING_TABLE_NAME,
    "chroma_path": CHROMA_DB_PATH,
    "chroma_mode": CHROMA_MODE,
    "collection_name": COLLECTION_NAME,
    "embedding_model": loaded_model_name,
    "chroma_init_mode": CHROMA_INIT_MODE,
    "chroma_init_errors": init_errors,
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log(f"Wrote runtime manifest to: {manifest_path}")



### Diagnostics Notes
- Each major code cell writes a diagnostics status line using `report_cell(...)`.
- If Chroma still fails due environment restrictions, embeddings are safely stored in persistent Delta.
- Share the final diagnostics summary printed in Cell 9 for troubleshooting.


In [ ]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)


def build_metadata(row):
    return {
        "act_name": safe_str(row.act_name),
        "section": safe_str(row.section_number),
        "category": safe_str(row.category),
        "source": safe_str(row.file_name),
    }


In [ ]:
print("[CELL 8] START - Building embeddings and writing Delta")

records = []
processed = 0
chroma_upserts = 0
batch_rows = []


def flush_batch(rows):
    texts = []
    ids = []
    metadatas = []

    for r in rows:
        chunk_text = safe_str(r.chunk_text).strip()
        chunk_id = safe_str(r.chunk_id).strip()

        if not chunk_text or not chunk_id:
            continue

        texts.append(chunk_text)
        ids.append(chunk_id)
        metadatas.append(build_metadata(r))

    if not ids:
        return 0, []

    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()

    ts = datetime.now(timezone.utc).isoformat()
    batch_records = []
    for i in range(len(ids)):
        batch_records.append({
            "chunk_id": ids[i],
            "chunk_text": texts[i],
            "act_name": metadatas[i]["act_name"],
            "section_number": metadatas[i]["section"],
            "category": metadatas[i]["category"],
            "file_name": metadatas[i]["source"],
            "embedding": embeddings[i],
            "embedding_model": loaded_model_name,
            "embedding_dim": len(embeddings[i]),
            "updated_at": ts,
        })

    upserted = 0
    if collection is not None:
        try:
            collection.upsert(
                ids=ids,
                documents=texts,
                embeddings=embeddings,
                metadatas=metadatas,
            )
            upserted = len(ids)
        except Exception as e:
            log(f"WARNING: Chroma upsert failed for current batch. Delta export will still continue. Error: {e}")

    return upserted, batch_records


for row in gold_df.toLocalIterator():
    batch_rows.append(row)

    if len(batch_rows) >= BATCH_SIZE:
        upserted, batch_records = flush_batch(batch_rows)
        chroma_upserts += upserted
        records.extend(batch_records)
        processed += len(batch_rows)
        log(f"Processed rows: {processed}")
        batch_rows = []

if batch_rows:
    upserted, batch_records = flush_batch(batch_rows)
    chroma_upserts += upserted
    records.extend(batch_records)
    processed += len(batch_rows)

if not records:
    report_cell("Cell 8 - Build Embeddings", False, "No embedding records were produced")
    print("[CELL 8] END - Failed: No records")
    raise RuntimeError("No embedding records were produced.")

embedding_df = spark.createDataFrame(records)
embedding_df = embedding_df.withColumn("updated_at", F.to_timestamp("updated_at"))

(
    embedding_df
    .dropDuplicates(["chunk_id"])
    .write
    .format("delta")
    .mode("overwrite")
    .save(EMBEDDING_DELTA_PATH)
)

# Optional table mirror for easier downstream reads
try:
    embedding_df.dropDuplicates(["chunk_id"]).write.mode("overwrite").saveAsTable(EMBEDDING_TABLE_NAME)
    table_status = f"updated table={EMBEDDING_TABLE_NAME}"
except Exception as e:
    table_status = f"table write skipped: {e}"

rows_written = embedding_df.count()
log(f"Delta embedding export complete at: {EMBEDDING_DELTA_PATH}")
log(f"Rows written to Delta: {rows_written}")
log(f"Vectors upserted to Chroma in this run: {chroma_upserts}")
if collection is not None:
    log(f"Current Chroma vector count: {collection.count()}")
log(table_status)

# Print latest Delta version info to confirm newest snapshot is active
try:
    hist = spark.sql(f"DESCRIBE HISTORY delta.`{EMBEDDING_DELTA_PATH}` LIMIT 1").collect()[0]
    latest_version = hist['version']
    latest_ts = hist['timestamp']
    print(f"Latest Delta version: {latest_version} | timestamp: {latest_ts}")
except Exception as e:
    print(f"Could not fetch Delta history: {e}")

report_cell("Cell 8 - Build Embeddings", True, f"delta_rows={rows_written}, chroma_upserts={chroma_upserts}, {table_status}")
print(f"[CELL 8] END - Success, delta_rows={rows_written}, chroma_upserts={chroma_upserts}")



In [ ]:
print("[CELL 9] START - Retrieval smoke test")

query = "What is the penalty for not wearing a helmet under Indian law?"
query_embedding = embedding_model.encode([query], show_progress_bar=False).tolist()[0]

if collection is not None:
    try:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=5,
        )

        docs = results.get("documents", [[]])[0]
        metas = results.get("metadatas", [[]])[0]

        print(f"Retrieved docs from Chroma: {len(docs)}")
        for i, (doc, meta) in enumerate(zip(docs, metas), start=1):
            print(f"\n--- Chroma Result {i} ---")
            print(meta)
            print(doc[:400])

        report_cell("Cell 9 - Retrieval Smoke Test", True, f"chroma_docs={len(docs)}, mode={CHROMA_MODE}")

    except Exception as e:
        report_cell("Cell 9 - Retrieval Smoke Test", False, f"Chroma query failed: {e}")

else:
    try:
        rows = spark.read.format("delta").load(EMBEDDING_DELTA_PATH).select(
            "chunk_text", "embedding", "act_name", "section_number"
        ).limit(2000).collect()

        import math
        def cosine(a, b):
            dot = sum(x * y for x, y in zip(a, b))
            na = math.sqrt(sum(x * x for x in a)) + 1e-12
            nb = math.sqrt(sum(x * x for x in b)) + 1e-12
            return dot / (na * nb)

        scored = []
        for r in rows:
            if not r.embedding:
                continue
            s = cosine(query_embedding, [float(x) for x in r.embedding])
            scored.append((s, r.chunk_text, r.act_name, r.section_number))

        scored.sort(key=lambda x: x[0], reverse=True)
        top = scored[:3]

        print(f"Delta fallback docs: {len(top)}")
        for i, item in enumerate(top, start=1):
            print(f"\n--- Delta Result {i} ---")
            print({"score": round(item[0], 4), "act_name": item[2], "section": item[3]})
            print((item[1] or "")[:400])

        report_cell("Cell 9 - Retrieval Smoke Test", True, f"delta_docs={len(top)}, chroma_unavailable=True")

    except Exception as e:
        report_cell("Cell 9 - Retrieval Smoke Test", False, f"Delta fallback failed: {e}")

print_diagnostics_summary()
print("[CELL 9] END")

